# 메시지 저장소: ChatMessageHistory
메시지 기록을 관리하는 객체로 어디에 저장느냐에 따라 여러 클래스들이 구현되어 제공된다.

## 종류
- **BaseChatMessageHistory**
    - 모든 메시지 기록 저장소 클래스의 **기본(최상위) 클래스**이다. 메시지를 저장하고 검색하는 기능을 정의하고 있으며, 이 클래스를 상속받아 다양한 저장소 방식이 구현된다.
	- `.add_user_message`, `.add_ai_message`
- **InMemoryChatMessageHistory**
    - 메시지를 **메모리에 저장**하는 방식이다. 속도가 빠르지만, 프로그램을 종료하면 저장된 메시지는 사라진다.
- 외부 저장소 연동 
    - Langchain은 다양한 **3rd-party 저장소**와 연동할 수 있다. 예를 들어 SQLite, PostgreSQL, Redis, MongoDB 등을 사용해 메시지를 영구적으로 저장할 수 있다.
    - https://python.langchain.com/docs/integrations/memory/

In [ ]:
#######################################
# ChatMessageHistory : 대화내역 저장소
#######################################


# memory에 저장
from langchain_core.chat_history import InMemoryChatMessageHistory
# role별 Message 객체
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage	

# ex)
# HumanMessage("내용") = ("user", "내용")
# AIMessage("내용") = ("ai", "내용")
# SystemMessage("내용") = ("system", "내용")


# 저장소 객체 생성
message_history = InMemoryChatMessageHistory()

# 저장소에 메세지 추가
message_history.?(SystemMessage("당신은 여행 가이드 입니다."))		# = ("system", "당신은 여행 가이드 입니다.")
message_history.?(HumanMessage("서울의 여행지 세 곳을 추천해줘용."))	# 질문
message_history.?(AIMessage("경북궁, 덕수궁, 창덕궁 가셈 ㅋㅋ"))		# 응답, .add_ai_message로도 가능.

In [ ]:
# 저장소에 저장된 대화 이력을 조회
message_history.?

In [ ]:
# DB에 저장 (SQLite)
# (참고) langchain_community : 3rd party 리소스/도구(외부 저장소)들과 연결하는 library
from langchain_community.chat_message_histories import SQLChatMessageHistory
from sqlalchemy import create_engine	# DB와 연결하기 위해

# SQLite
engine = create_engine("sqlite:///message_history.sqlite")


sql_message_history = SQLChatMessageHistory(
    session_id="user_1",		# 대화내역을 저장할 사용자 ID(구분자)
    connection=?				# 만들어 놓은 engine과 연결 !
)

# add_"role"_message로 user, ai message 추가 가능 !
## BaseChatMessageHistory class에서 지원하는 method !
sql_message_history.add_user_message("안녕하세요")
sql_message_history.add_ai_message("안녕하세요, 어떻게 도와드릴깝쇼 ?!?!")
sql_message_history.add_user_message("이름이 뭐에여 ? 저나버너뭐에여 ?")

In [ ]:
sql_message_history.?

# RunnableWithMessageHistory
## 생성

`RunnableWithMessageHistory`는 다음과 같은 요소들을 initializer에 전달해 생성한다.

- **runnable**: 실제 작업을 수행하는 체인(`Runnable`) 객체이다.
- **get_session_history**: 주어진 `session_id`에 해당하는 메시지 기록 저장소(`ChatMessageHistory`) 객체를 반환하는 함수이다.
- **input_messages_key**: 사용자 입력 메시지를 저장할 입력 필드의 이름이다.
- **history_messages_key**: 저장된 이전 대화 메시지를 불러올 필드의 이름이다.

이를 통해 체인을 실행할 때마다 이전 메시지가 자동으로 전달되고, 새로운 메시지도 기록된다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from dotenv import load_dotenv

load_dotenv()

In [ ]:
prompt_template = ChatPromptTemplate(
    [
        ("system", ("당신은 AI 분야 전문가야. "
         "전문가 스타일로 답변해 주시길. "
         "답변은 20단어 이내로 설명 부탁. "
         "정확하지 않은 경우 모른다고 꼭 말해주시길.")),
        ?(variable_name="history", optional=True),	# = ("placeholder", "{history}")
        ("human", "{query}")
	]
)

model = ChatOpenAI(model_name = "gpt-4o-mini")
chain = prompt_template | model

In [ ]:
# key: session_id, Value: InMemoryChatMessageHistory객체 -> session_id별로 따로 저장소를 생성해서 관리
# InMemoryChatMessageHistory는 session_id별로 대화를 관리하는 기능이 없음. 만들어줘야함
store = {}
def get_session_history(session_id : str) -> InMemoryChatMessageHistory:
    # user의 session_id를 받아서 그 session id의 대화를 관리하는 ChatMessageHistory를 반환.
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [ ]:
chain_with_history = RunnableWithMessageHistory(
    runnable=?,					# 실제 대화를 처리할 chain (prompt_template -> model -> parser(optional))
    get_session_history=?,		# Session_id를 받아서 그 사용자의 대화를 관리하는 대화 저장소를 제공하는 func or callable
    input_messages_key="?",		# 사용자 질문을 넣을 PromptTemplate의 변수 이름.
    history_messages_key="?"	# 대화이력(저장소에서 조회한)을 넣을 PromptTemplate의 변수 이름.
)

In [ ]:
config = {"configurable" : {"session_id" : "conv-1"}}

while True:
    query = input("User Prompt")
    query_dict = {"query" : query}
    if query == "!q":
        print(">>>> 대화 끗")
        break
    res = chain_with_history.invoke(query_dict, config)
    print(f">>>>> User : {query}")
    print(f"<<<<< AI : {res.content}")

In [ ]:
store